# Specialized LLMs: When General-Purpose Models Aren't Enough

## Why Domain-Specific Models Exist

General-purpose LLMs like GPT-4, Claude, or Llama are trained on broad internet data. They are remarkably capable across many tasks but they have blind spots:

- **Rare domain vocabulary**: Medical ICD codes, legal citations, SMILES molecular strings, financial ticker conventions
- **Specialized reasoning patterns**: Formal mathematical proofs, differential diagnosis logic, code execution semantics
- **Data distribution mismatch**: PubMed abstracts, court opinions, and Bloomberg terminals are a tiny fraction of Common Crawl
- **Safety and compliance requirements**: HIPAA in healthcare, regulatory constraints in finance and law

Domain-specific LLMs address these gaps by **pretraining or fine-tuning on curated domain corpora**, often yielding dramatically better performance on benchmarks despite being much smaller than frontier general models.

## Trade-offs: Specialized vs. General Models

| Dimension | General-Purpose LLM | Domain-Specific LLM |
|-----------|--------------------|-----------------------|
| **Domain accuracy** | Moderate | High (in-domain) |
| **Out-of-domain flexibility** | High | Low (catastrophic forgetting risk) |
| **Training cost** | Enormous | Moderate (DAPT or fine-tune) |
| **Data requirements** | Trillions of tokens | Billions of domain tokens |
| **Maintenance** | Single model | Per-domain model zoo |
| **Hallucination risk** | Present | Still present; critical in medicine/law |
| **Compliance** | Often unaudited | Trainable on compliant data |

> **Key insight**: A 7B specialized model can outperform a 70B general model on domain-specific benchmarks. Size is not the only lever.

---
## 1. Code LLMs

Code LLMs are perhaps the most commercially successful category of specialized models. They are trained on massive repositories of source code (GitHub, GitLab, Bitbucket, etc.) and understand programming languages at a structural and semantic level.

### Major Code LLMs

| Model | Organization | Key Features |
|-------|-------------|-------------|
| **Code Llama** | Meta | 7/13/34/70B, infilling (PSM), Python-specialized variant |
| **DeepSeek-Coder** | DeepSeek AI | Fill-in-Middle (FIM), repo-level context window (16K), V2 series |
| **StarCoder / StarCoder2** | BigCode | The Stack dataset, 80+ languages, 15B params, open weights |
| **CodeT5 / CodeT5+** | Salesforce | Encoder-decoder (T5 architecture), code understanding + generation |
| **SantaCoder** | BigCode | 1.1B, strong multilingual baseline |
| **WizardCoder** | WizardLM | Evol-Instruct applied to code data |
| **Phind-CodeLlama** | Phind | Fine-tuned Code Llama with proprietary code dataset |
| **CodeGemma** | Google | 2B/7B, built on Gemma, 500B token code training |
| **OpenCodeInterpreter** | OpenCSG | Code generation + execution feedback loop |
| **AlphaCode / AlphaCode 2** | DeepMind | Competitive programming, test-based filtering, 50th percentile Codeforces |
| **GitHub Copilot (Codex)** | OpenAI/GitHub | Commercial product, GPT-4 backbone, IDE integration |

### Infilling Objectives

A key capability of code LLMs is **fill-in-the-middle (FIM)** predicting a middle span given both prefix and suffix context. This is essential for IDE completions.

**PSM (Prefix-Suffix-Middle)** format used by Code Llama:

$$\text{input} = \texttt{<PRE>} \; x_{prefix} \; \texttt{<SUF>} \; x_{suffix} \; \texttt{<MID>}$$

**SPM (Suffix-Prefix-Middle)** alternative ordering:

$$\text{input} = \texttt{<SUF>} \; x_{suffix} \; \texttt{<PRE>} \; x_{prefix} \; \texttt{<MID>}$$

The training objective randomly masks a span and trains the model to predict it this teaches bidirectional context awareness without an encoder.

### Benchmarks

- **HumanEval** (OpenAI, 2021): 164 Python problems, pass@k metric
- **MBPP** (Google, 2021): 374 crowd-sourced Python problems
- **SWE-bench** (2023): Real GitHub issues requiring multi-file repository edits much harder than HumanEval

In [1]:
# Skip heavy model loading in automated execution
import os
if not os.environ.get("RUN_HEAVY_MODELS"):
    print("[Skipped: requires model download. Set RUN_HEAVY_MODELS=1 to run]")
else:
    # Code LLM: Fill-in-the-Middle with Code Llama
    # Requires: pip install transformers accelerate torch
    # Model download: ~13GB for codellama/CodeLlama-7b-hf

    from transformers import AutoTokenizer, AutoModelForCausalLM
    import torch

    # ── Load model and tokenizer ──────────────────────────────────────────────────
    model_id = "codellama/CodeLlama-7b-hf"  # base model supports FIM
    # For instruction following, use: "codellama/CodeLlama-7b-Instruct-hf"

    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.float16,   # use bf16 on Ampere+ GPUs
        device_map="auto"            # automatically split across available GPUs
    )

    # ── Fill-in-the-Middle (FIM / PSM format) ────────────────────────────────────
    # Code Llama uses special tokens: <PRE>, <SUF>, <MID>, <EOT>
    # The model predicts the middle section between prefix and suffix.

    prefix = """def fibonacci(n: int) -> int:
        \"\"\"Return the nth Fibonacci number.\"\"\"
        if n <= 1:
            return n
    """

    suffix = """
        return fibonacci(n - 1) + fibonacci(n - 2)
    """

    # Construct the PSM prompt with special infilling tokens
    fim_prompt = f"<PRE> {prefix} <SUF>{suffix} <MID>"

    inputs = tokenizer(fim_prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=64,
            temperature=0.2,       # low temperature for deterministic code
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )

    # Decode only the newly generated tokens (not the prompt)
    generated = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )
    print("=== Infilled middle section ===")
    print(generated)

[Skipped: requires model download. Set RUN_HEAVY_MODELS=1 to run]


In [2]:
# ── Code Llama Instruct: chat-style code generation ──────────────────────────
# The Instruct variant uses a special [INST] / [/INST] prompt format

instruct_model_id = "codellama/CodeLlama-7b-Instruct-hf"
# tokenizer and model loading same as above...

def code_llama_instruct_prompt(system: str, user: str) -> str:
    """Build the Code Llama instruct prompt format."""
    return f"""<s>[INST] <<SYS>>
{system}
<</SYS>>

{user} [/INST]"""

prompt = code_llama_instruct_prompt(
    system="You are a helpful coding assistant. Provide clean, well-commented Python code.",
    user="Write a Python function that implements binary search on a sorted list."
)

print("=== Prompt sent to Code Llama Instruct ===")
print(prompt)

# ── DeepSeek-Coder FIM format (for comparison) ───────────────────────────────
# DeepSeek-Coder uses different special tokens:
# <｜fim▁begin｜>  <｜fim▁hole｜>  <｜fim▁end｜>

ds_prefix = "def quick_sort(arr: list) -> list:\n    "
ds_suffix = "\n    return arr"

# DeepSeek FIM format
ds_fim_prompt = f"<｜fim▁begin｜>{ds_prefix}<｜fim▁hole｜>{ds_suffix}<｜fim▁end｜>"
print("\n=== DeepSeek-Coder FIM prompt ===")
print(ds_fim_prompt)

=== Prompt sent to Code Llama Instruct ===
<s>[INST] <<SYS>>
You are a helpful coding assistant. Provide clean, well-commented Python code.
<</SYS>>

Write a Python function that implements binary search on a sorted list. [/INST]

=== DeepSeek-Coder FIM prompt ===
<｜fim▁begin｜>def quick_sort(arr: list) -> list:
    <｜fim▁hole｜>
    return arr<｜fim▁end｜>


---
## 2. Math LLMs

Mathematics requires **exact symbolic reasoning**, multi-step deduction, and verification skills that are notoriously difficult for neural networks trained on text prediction. Specialized math LLMs close this gap through better training data and objectives.

### Major Math LLMs

| Model | Key Innovation |
|-------|---------------|
| **Llemma** (EleutherAI) | Continued pretraining on Proof-Pile-2 (55B math tokens: ArXiv, math web pages, GitHub proofs) |
| **MetaMath** (Meta) | Data augmentation: rewrite GSM8K/MATH problems from different angles (backward reasoning, self-verification) |
| **MAmmoTH** (TIGER-Lab) | Chain-of-thought + tool-integrated reasoning (Python interpreter) on MathInstruct |
| **WizardMath** | Evol-Instruct applied to math: progressively harder problem variants |
| **DeepSeek-Math** | 7B model trained on 120B math tokens, GRPO reinforcement learning |
| **Qwen2.5-Math** | 72B, strong on Chinese and English math olympiad problems |
| **NuminaMath** | Focused on competition mathematics (AMC, AIME, IMO problems) |

### Chain-of-Thought as Structured Reasoning

Math LLMs leverage **chain-of-thought (CoT)** prompting. Formally, instead of predicting the answer directly:

$$P(a \mid q) \approx \sum_{z} P(a \mid q, z) \cdot P(z \mid q)$$

where $z = (z_1, z_2, \ldots, z_k)$ is a sequence of intermediate reasoning steps. The model learns to generate $z$ before $a$, which dramatically improves accuracy on multi-step problems.

**Tool-Augmented Reasoning** (MAmmoTH approach):

$$z_i = \begin{cases} \text{text step} & \text{if } z_i \text{ is natural language} \\ \texttt{exec}(\text{Python code}) & \text{if } z_i \text{ is a code block} \end{cases}$$

### Benchmarks

- **GSM8K**: 8,500 grade-school math word problems tests arithmetic reasoning
- **MATH**: 12,500 competition math problems (AMC/AIME level) much harder, tests algebraic/geometric reasoning
- **MGSM**: Multilingual GSM8K (10 languages)

In [3]:
# Math LLM: Chain-of-Thought prompting with WizardMath / DeepSeek-Math
# This demonstrates the prompting pattern; actual inference requires model download.

from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch

# ── Option A: WizardMath (fine-tuned on math instruction data) ────────────────
model_id = "WizardLM/WizardMath-7B-V1.1"

# WizardMath uses a specific system prompt that elicits CoT
WIZARDMATH_SYSTEM = (
    "Below is an instruction that describes a task. "
    "Write a response that appropriately completes the request.\n\n"
)

def build_math_prompt(problem: str) -> str:
    """Construct WizardMath-style prompt with CoT instruction."""
    return (
        f"{WIZARDMATH_SYSTEM}"
        f"### Instruction:\n{problem}\n\n"
        f"### Response: Let's think step by step.\n"
    )

# Example math problem
problem = (
    "A train travels from City A to City B at 60 km/h and returns at 40 km/h. "
    "What is the average speed for the entire trip?"
)

prompt = build_math_prompt(problem)
print("=== Math Prompt ===")
print(prompt)
print()

# ── Option B: DeepSeek-Math with GRPO-trained reasoning ──────────────────────
# DeepSeek-Math-7B-Instruct uses a simpler format
def deepseek_math_prompt(problem: str) -> str:
    """DeepSeek-Math instruction format."""
    return (
        f"User: {problem}\n"
        f"Please reason step by step, and put your final answer within \\boxed{{}}.\n"
        f"Assistant:"
    )

ds_prompt = deepseek_math_prompt(problem)
print("=== DeepSeek-Math Prompt ===")
print(ds_prompt)

# ── Simulated CoT output (what a well-trained model would produce) ─────────────
expected_cot = """
Let d be the one-way distance.

Step 1: Time from A to B = d / 60
Step 2: Time from B to A = d / 40
Step 3: Total distance = 2d
Step 4: Total time = d/60 + d/40 = 2d/120 + 3d/120 = 5d/120 = d/24
Step 5: Average speed = Total distance / Total time = 2d / (d/24) = 48 km/h

Answer: \\boxed{48} km/h  (harmonic mean of 60 and 40)
"""
print("=== Expected CoT Output ===")
print(expected_cot)

/home/dell/Desktop/AI_Tasks/Additional_Data/zero-to-ai-engineer/zero-to-ai-engineer/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


=== Math Prompt ===
Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
A train travels from City A to City B at 60 km/h and returns at 40 km/h. What is the average speed for the entire trip?

### Response: Let's think step by step.


=== DeepSeek-Math Prompt ===
User: A train travels from City A to City B at 60 km/h and returns at 40 km/h. What is the average speed for the entire trip?
Please reason step by step, and put your final answer within \boxed{}.
Assistant:
=== Expected CoT Output ===

Let d be the one-way distance.

Step 1: Time from A to B = d / 60
Step 2: Time from B to A = d / 40
Step 3: Total distance = 2d
Step 4: Total time = d/60 + d/40 = 2d/120 + 3d/120 = 5d/120 = d/24
Step 5: Average speed = Total distance / Total time = 2d / (d/24) = 48 km/h

Answer: \boxed{48} km/h  (harmonic mean of 60 and 40)



---
## 3. Medical / Healthcare LLMs

Medical AI is one of the highest-stakes domains for LLMs. Errors can directly harm patients, making accuracy, calibration, and auditability paramount.

### Major Medical LLMs

| Model | Organization | Training Data / Focus |
|-------|-------------|----------------------|
| **Med-PaLM** | Google | Instruction-tuned PaLM on MultiMedQA; first model to pass USMLE |
| **Med-PaLM 2** | Google | Expert-level performance on USMLE (86.5%), based on PaLM 2 |
| **Meditron-70B** | EPFL | LLaMA 2 fine-tuned on medical guidelines (PubMed, medical guidelines corpus) |
| **BioGPT** | Microsoft | GPT-2 scale, trained on PubMed abstracts; biomedical text mining |
| **PubMedBERT** | Microsoft | BERT pretrained from scratch on PubMed (no general pretraining) |
| **BioBERT** | DMIS Lab | BERT fine-tuned on PubMed + PMC full texts |
| **GatorTron** | UF Health | 8.9B param transformer trained on 90B+ clinical notes (UF Health EHR) |
| **ClinicalBERT** | MIT | BERT fine-tuned on MIMIC-III clinical notes |
| **PMC-LLaMA** | PKU | LLaMA fine-tuned on PubMed Central full texts |

### Critical Challenges

**1. Hallucination danger**: A model confidently citing a non-existent drug interaction can cause patient harm. Calibration and retrieval-augmented generation (RAG) are essential.

**2. HIPAA compliance (US)**: Protected Health Information (PHI) patient names, dates, locations, diagnoses cannot be used for training without de-identification.

**3. Clinical heterogeneity**: The same concept appears differently across hospital systems, specialties, and countries (ICD-10-CM vs ICD-11, SNOMED CT, LOINC, etc.).

**4. Temporal distribution shift**: Drug approvals, clinical guidelines, and best practices change a model trained in 2022 may recommend outdated treatments.

### Benchmarks

- **MedQA (USMLE)**: 12K+ multiple-choice questions from US Medical Licensing Exam
- **PubMedQA**: 211K biomedical questions with context from PubMed abstracts
- **MultiMedQA**: Google's aggregate of MedQA, MedMCQA, PubMedQA, LiveQA-Med, and Medication QA

In [4]:
# Skip heavy model loading in automated execution
import os
if not os.environ.get("RUN_HEAVY_MODELS"):
    print("[Skipped: requires model download. Set RUN_HEAVY_MODELS=1 to run]")
else:
    # Medical NLP: PubMedBERT for Biomedical Named Entity Recognition (NER)
    # Requires: pip install transformers torch

    from transformers import pipeline, AutoTokenizer, AutoModelForTokenClassification

    # ── PubMedBERT fine-tuned for NER ─────────────────────────────────────────────
    # This model is fine-tuned on NCBI-disease corpus for disease entity recognition.
    # microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext is the base.
    # pruas/BENT-PubMedBERT-NER-Disease is a fine-tuned NER variant.

    ner_model_id = "pruas/BENT-PubMedBERT-NER-Disease"

    ner_pipeline = pipeline(
        task="ner",
        model=ner_model_id,
        aggregation_strategy="simple",  # merge subword tokens into words
        device=0 if __import__('torch').cuda.is_available() else -1
    )

    # Sample biomedical text
    clinical_text = (
        "The patient presented with symptoms consistent with type 2 diabetes mellitus "
        "and was subsequently diagnosed with diabetic retinopathy. "
        "Family history was positive for hypertension and coronary artery disease."
    )

    entities = ner_pipeline(clinical_text)

    print("=== Biomedical NER Results ===")
    print(f"Input: {clinical_text}\n")
    for entity in entities:
        print(
            f"  Entity: {entity['word']:<35} "
            f"Label: {entity['entity_group']:<15} "
            f"Score: {entity['score']:.3f}"
        )

[Skipped: requires model download. Set RUN_HEAVY_MODELS=1 to run]


In [5]:
# Skip heavy model loading in automated execution
import os
if not os.environ.get("RUN_HEAVY_MODELS"):
    print("[Skipped: requires model download. Set RUN_HEAVY_MODELS=1 to run]")
else:
    # Medical QA: BioGPT for biomedical question answering
    from transformers import pipeline

    # BioGPT-Large fine-tuned on PubMedQA
    biogpt_qa = pipeline(
        "text-generation",
        model="microsoft/biogpt",   # base BioGPT; biogpt-large is also available
        max_new_tokens=100,
        do_sample=False            # greedy decoding for factual QA
    )

    # BioGPT was trained to answer questions given a context passage
    context = (
        "Metformin is a biguanide class medication commonly used as first-line "
        "pharmacological therapy for type 2 diabetes. "
        "It works primarily by decreasing hepatic glucose production."
    )
    question = "What is the primary mechanism of action of Metformin?"

    # BioGPT format: question followed by context
    biogpt_input = f"{question} {context}"

    result = biogpt_qa(biogpt_input)
    print("=== BioGPT Medical QA ===")
    print(f"Question: {question}")
    print(f"Context: {context[:80]}...")
    print(f"Answer: {result[0]['generated_text'][len(biogpt_input):].strip()}")

    # ── Important warning for production medical AI ───────────────────────────────
    print("\n" + "="*60)
    print("WARNING: LLM outputs should NEVER replace clinical judgment.")
    print("Always validate with qualified medical professionals and")
    print("up-to-date clinical guidelines.")
    print("="*60)

[Skipped: requires model download. Set RUN_HEAVY_MODELS=1 to run]


---
## 4. Legal LLMs

Legal language is highly specialized: statutes, case law citations, terms of art with precise meanings, and jurisdiction-specific interpretations. General models often miss nuances that a trained lawyer would catch immediately.

### Major Legal LLMs

| Model | Key Details |
|-------|------------|
| **Legal-BERT** | BERT fine-tuned on EU legislation, ECJ cases, contracts; strong on LexGLUE |
| **CaseLawBERT** | BERT trained on Harvard Caselaw Access Project (~3.5M US case opinions) |
| **SaulLM-7B / SaulLM-54B** | Mistral-based, trained on 30B tokens of English legal text (Euipia corpus); best open-source legal LLM as of 2024 |
| **Lawyer LLaMA** | LLaMA fine-tuned with Chinese legal knowledge; QA + consultation |
| **ChatLaw** | Chinese law-focused; integrates legal knowledge base via RAG |

### Key Challenges in Legal NLP

**1. Document length**: Contracts, case opinions, and statutory compilations can be 100K+ tokens. Long-context models (LongFormer, BigBird) or chunking strategies are required.

**2. Jurisdiction specificity**: "Negligence" has different legal standards across common law vs. civil law jurisdictions, and even between US states.

**3. Citation conventions**: Legal writing is dense with cross-references ("See *Brown v. Board of Education*, 347 U.S. 483 (1954)") that require structured knowledge.

**4. Non-standard language**: Legal writing uses archaic terms (*inter alia*, *res judicata*, *mens rea*) and deliberately ambiguous phrasing that creates interpretive challenges.

**5. Ethical constraints**: Unauthorized practice of law (UPL) regulations vary by jurisdiction; AI legal tools must clearly disclaim they are not providing legal advice.

### Benchmark: LexGLUE

LexGLUE (Legal General Language Understanding Evaluation) covers 7 datasets:
- ECtHR (European Court of Human Rights case outcome prediction)
- SCOTUS (US Supreme Court decision topic classification)
- EUR-Lex (EU legislation classification)
- LEDGAR (contract clause type classification)
- UNFAIR-ToS (unfair clause detection in Terms of Service)
- CaseHOLD (legal holding identification)

In [6]:
# Skip heavy model loading in automated execution
import os
if not os.environ.get("RUN_HEAVY_MODELS"):
    print("[Skipped: requires model download. Set RUN_HEAVY_MODELS=1 to run]")
else:
    # Legal NLP: Legal-BERT for legal text classification
    # Demonstrating contract clause type classification (LEDGAR task)

    from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification
    import torch

    # ── Legal-BERT for sequence classification ────────────────────────────────────
    # nlpaueb/legal-bert-base-uncased is the base model
    # For LEDGAR clause classification, we use a fine-tuned version
    legal_clf = pipeline(
        "text-classification",
        model="nlpaueb/legal-bert-base-uncased",
        # In practice, you'd fine-tune this on your specific legal classification task
        device=0 if torch.cuda.is_available() else -1,
        top_k=3  # return top-3 predictions
    )

    # Sample contract clauses for classification
    clauses = [
        # Indemnification clause
        ("Indemnification",
         "Each party shall indemnify, defend, and hold harmless the other party "
         "from and against any claims, damages, losses, costs, and expenses "
         "arising from its breach of this Agreement."),

        # Governing law clause
        ("Governing Law",
         "This Agreement shall be governed by and construed in accordance with "
         "the laws of the State of Delaware, without regard to its conflict of "
         "law principles."),

        # Limitation of liability clause
        ("Limitation of Liability",
         "IN NO EVENT SHALL EITHER PARTY BE LIABLE FOR ANY INDIRECT, INCIDENTAL, "
         "SPECIAL, EXEMPLARY, OR CONSEQUENTIAL DAMAGES, HOWEVER CAUSED, EVEN IF "
         "ADVISED OF THE POSSIBILITY OF SUCH DAMAGES."),
    ]

    print("=== Legal Clause Classification ===")
    for expected_type, clause_text in clauses:
        print(f"\nClause type (expected): {expected_type}")
        print(f"Text: {clause_text[:80]}...")
        try:
            predictions = legal_clf(clause_text)
            for pred in predictions[0]:
                print(f"  → {pred['label']}: {pred['score']:.3f}")
        except Exception as e:
            print(f"  [Model not loaded prediction would appear here: {e}]")

    # ── SaulLM usage example (generative, much larger) ────────────────────────────
    print("\n=== SaulLM-7B Prompt Format ===")
    saullm_prompt = """[INST] Analyze the following contract clause and identify potential risks:

    "The licensor reserves the right to modify or discontinue the service at any time 
    without notice and without liability to the licensee."

    Provide a structured legal analysis. [/INST]"""
    print(saullm_prompt)
    print("\n[SaulLM-7B would generate a structured legal analysis here]")

[Skipped: requires model download. Set RUN_HEAVY_MODELS=1 to run]


---
## 5. Scientific LLMs

Scientific literature is one of the fastest-growing text corpora, yet it requires deep domain understanding specialized notation, citation practices, experimental methodology, and field-specific ontologies.

### Major Scientific LLMs

| Model | Domain / Key Feature |
|-------|---------------------|
| **Galactica** (Meta, 120B) | Scientific knowledge synthesis, citation generation, LaTeX output controversial due to confident hallucination |
| **SciBERT** (AllenAI) | BERT trained on 1.14M papers from Semantic Scholar (18% CS, 82% biomedical) |
| **S2ORC** | Semantic Scholar Open Research Corpus 81.1M papers, used as training data |
| **OLMo** (AllenAI) | Fully open: training data, code, checkpoints, logs; Dolma corpus |
| **AstroBERT** | BERT trained on NASA ADS astrophysics papers |
| **ChemBERTa** | RoBERTa trained on SMILES molecular strings (10M molecules from PubChem) |
| **MatSciBERT** | BERT trained on materials science literature (abstracts + full texts) |

### SMILES Notation (Chemistry)

ChemBERTa treats SMILES strings as a language. SMILES (Simplified Molecular Input Line Entry System) encodes molecular structure as text:

```
Aspirin:    CC(=O)Oc1ccccc1C(=O)O
Caffeine:   Cn1cnc2c1c(=O)n(c(=O)n2C)C
Benzene:    c1ccccc1
```

By tokenizing SMILES with a custom vocabulary and applying transformer pretraining, ChemBERTa learns representations that capture chemical properties like solubility, toxicity, and bioactivity.

### Benchmarks

- **SciEval** / **SciEx**: Multi-domain scientific QA and reasoning
- **MoleculeNet**: Molecular property prediction benchmarks
- **BLURB** (Biomedical Language Understanding and Reasoning Benchmark)

In [7]:
# Skip heavy model loading in automated execution
import os
if not os.environ.get("RUN_HEAVY_MODELS"):
    print("[Skipped: requires model download. Set RUN_HEAVY_MODELS=1 to run]")
else:
    # Scientific NLP: SciBERT for scientific text classification
    # and ChemBERTa for molecular property prediction

    from transformers import AutoTokenizer, AutoModel, pipeline
    import torch
    import numpy as np

    # ── SciBERT: Extract scientific text embeddings ───────────────────────────────
    scibert_tokenizer = AutoTokenizer.from_pretrained("allenai/scibert_scivocab_uncased")
    scibert_model = AutoModel.from_pretrained("allenai/scibert_scivocab_uncased")
    scibert_model.eval()

    def get_scibert_embedding(text: str) -> np.ndarray:
        """Get CLS token embedding from SciBERT for a scientific text."""
        inputs = scibert_tokenizer(
            text,
            return_tensors="pt",
            max_length=512,
            truncation=True,
            padding=True
        )
        with torch.no_grad():
            outputs = scibert_model(**inputs)
        # CLS token = first token of last hidden state
        cls_embedding = outputs.last_hidden_state[:, 0, :].squeeze().numpy()
        return cls_embedding

    # Compare embeddings of related vs. unrelated abstracts
    abstracts = {
        "transformer_nlp": (
            "Attention mechanisms allow the model to focus on relevant parts of the input "
            "sequence when generating each output token."
        ),
        "protein_folding": (
            "The three-dimensional structure of proteins determines their biochemical function. "
            "AlphaFold uses attention-based neural networks to predict protein structure."
        ),
        "black_hole_physics": (
            "Gravitational wave observations from LIGO confirm predictions of general relativity "
            "regarding black hole mergers."
        ),
    }

    embeddings = {k: get_scibert_embedding(v) for k, v in abstracts.items()}

    # Compute cosine similarities
    def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
        return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

    keys = list(embeddings.keys())
    print("=== SciBERT Embedding Cosine Similarities ===")
    for i in range(len(keys)):
        for j in range(i + 1, len(keys)):
            sim = cosine_similarity(embeddings[keys[i]], embeddings[keys[j]])
            print(f"  {keys[i]} <-> {keys[j]}: {sim:.4f}")

    # Expected: transformer_nlp <-> protein_folding > transformer_nlp <-> black_hole_physics
    # (because AlphaFold uses transformers SciBERT should capture this cross-domain link)

[Skipped: requires model download. Set RUN_HEAVY_MODELS=1 to run]


In [8]:
# Skip heavy model loading in automated execution
import os
if not os.environ.get("RUN_HEAVY_MODELS"):
    print("[Skipped: requires model download. Set RUN_HEAVY_MODELS=1 to run]")
else:
    # ChemBERTa: Molecular representation learning from SMILES
    from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
    import torch

    # ── ChemBERTa for molecular property prediction ───────────────────────────────
    # seyonec/ChemBERTa-zinc-base-v1: pretrained on 100K ZINC molecules
    # seyonec/PubChem10M_SMILES_BPE_450k: pretrained on 10M PubChem molecules

    chemberta_tokenizer = AutoTokenizer.from_pretrained("seyonec/ChemBERTa-zinc-base-v1")
    chemberta_model = AutoModel.from_pretrained("seyonec/ChemBERTa-zinc-base-v1")
    chemberta_model.eval()

    # Well-known molecules as SMILES strings
    molecules = {
        "Aspirin": "CC(=O)Oc1ccccc1C(=O)O",
        "Caffeine": "Cn1cnc2c1c(=O)n(c(=O)n2C)C",
        "Ethanol": "CCO",
        "Benzene": "c1ccccc1",
        "Glucose": "C(C1C(C(C(C(O1)O)O)O)O)O",
        "Penicillin G": "CC1(C(N2C(S1)C(C2=O)NC(=O)Cc3ccccc3)C(=O)O)C",
    }

    def get_molecular_embedding(smiles: str) -> torch.Tensor:
        """Get CLS embedding for a SMILES string using ChemBERTa."""
        inputs = chemberta_tokenizer(
            smiles,
            return_tensors="pt",
            max_length=512,
            truncation=True
        )
        with torch.no_grad():
            outputs = chemberta_model(**inputs)
        return outputs.last_hidden_state[:, 0, :]  # CLS token

    print("=== ChemBERTa: Molecular Embeddings ===")
    print(f"{'Molecule':<15} {'SMILES':<45} {'Embedding dim'}")
    print("-" * 75)
    for name, smiles in molecules.items():
        emb = get_molecular_embedding(smiles)
        print(f"{name:<15} {smiles:<45} {emb.shape[-1]}")

    print("\nThese embeddings can be used for:")
    print("  - Molecular property prediction (solubility, toxicity, bioactivity)")
    print("  - Virtual screening (similarity search over chemical space)")
    print("  - Drug-target interaction prediction")
    print("  - Chemical reaction prediction")

[Skipped: requires model download. Set RUN_HEAVY_MODELS=1 to run]


---
## 6. Financial LLMs

Finance presents unique challenges: structured numerical data mixed with unstructured text, time-sensitive information, regulatory requirements, and the direct economic consequences of errors.

### Major Financial LLMs

| Model | Params | Key Details |
|-------|--------|------------|
| **BloombergGPT** | 50B | Trained on 363B tokens of Bloomberg financial data + 345B general tokens; decoder-only |
| **FinGPT** | 7B-13B | Open-source; uses LoRA fine-tuning on financial instruction data; reproducible |
| **PIXIU** | 7B-13B | Financial instruction tuning benchmark + models; covers 8 tasks |
| **XuanYuan** | 70B | Chinese financial LLM from DFCF; Chinese financial news, reports, QA |
| **FinBERT** | 110M | BERT fine-tuned on Financial PhraseBank for sentiment analysis |

### BloombergGPT Architecture Details

BloombergGPT (2023) was the first large-scale domain-specific financial LLM. Key design decisions:

- **Mixed training**: 50/50 split between Bloomberg financial corpus and general text (The Pile, C4)
- **Financial corpus**: Bloomberg News (21 years), filings, press releases, web scraped financial content
- **Tokenizer**: Custom BPE vocabulary trained on financial text (optimized for tickers, financial terms)
- **Architecture**: BLOOM-style decoder-only, 70 layers, 40 attention heads

**Key finding**: BloombergGPT outperforms similar-sized general models (GPT-NeoX-20B) on financial tasks while maintaining competitive general performance demonstrating that domain data enhances without catastrophically forgetting.

### Financial Tasks

| Task | Description | Key Models |
|------|-------------|----------|
| Sentiment Analysis | Bullish/bearish/neutral on news | FinBERT, BloombergGPT |
| Named Entity Recognition | Tickers, companies, monetary values | FinBERT-NER |
| Question Answering | Financial report QA | FinQA dataset |
| Summarization | Earnings call summaries | BloombergGPT, FinGPT |
| Numerical Reasoning | Revenue calculation, percentage changes | FinQA, TAT-QA |

### Benchmark: FinQA

FinQA requires numerical reasoning over financial reports models must extract numbers from tables and text, then perform multi-step arithmetic. This is significantly harder than pure text QA.

In [9]:
# Skip heavy model loading in automated execution
import os
if not os.environ.get("RUN_HEAVY_MODELS"):
    print("[Skipped: requires model download. Set RUN_HEAVY_MODELS=1 to run]")
else:
    # Financial NLP: FinBERT for financial sentiment analysis
    # FinBERT is one of the most widely used financial NLP models

    from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification
    import torch

    # ── Load FinBERT sentiment classifier ────────────────────────────────────────
    # ProsusAI/finbert is fine-tuned on Financial PhraseBank
    # Labels: positive, negative, neutral

    finbert = pipeline(
        "text-classification",
        model="ProsusAI/finbert",
        tokenizer="ProsusAI/finbert",
        device=0 if torch.cuda.is_available() else -1,
        top_k=None  # return all labels with scores
    )

    # Representative financial news headlines
    headlines = [
        "Apple reports record quarterly earnings, beating analyst expectations by 15%",
        "Federal Reserve signals additional rate hikes amid persistent inflation concerns",
        "Tech startup announces layoffs of 20% of workforce following funding shortfall",
        "Merger between two regional banks approved by regulators after 18-month review",
        "Oil prices stabilize after OPEC+ maintains current production levels",
        "Cryptocurrency exchange files for bankruptcy protection amid liquidity crisis",
    ]

    print("=== FinBERT Financial Sentiment Analysis ===")
    print(f"{'Headline':<60} {'Sentiment':<12} {'Confidence'}")
    print("-" * 85)

    for headline in headlines:
        result = finbert(headline)
        # Get top prediction
        top = max(result[0], key=lambda x: x['score'])
        short_headline = headline[:57] + "..." if len(headline) > 57 else headline
        print(f"{short_headline:<60} {top['label']:<12} {top['score']:.3f}")

[Skipped: requires model download. Set RUN_HEAVY_MODELS=1 to run]


In [10]:
# ── FinGPT: Open-source financial LLM with LoRA ───────────────────────────────
# FinGPT demonstrates how to build competitive financial LLMs cheaply using LoRA

# FinGPT training approach (pseudo-code showing the LoRA fine-tuning recipe)
from peft import LoraConfig, get_peft_model, TaskType  # pip install peft
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer

# Step 1: Load a base model (LLaMA-2 or Falcon)
base_model_id = "meta-llama/Llama-2-7b-hf"  # or "tiiuae/falcon-7b"

# Step 2: Configure LoRA
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,                           # LoRA rank (low-rank approximation dimension)
    lora_alpha=32,                 # scaling factor
    target_modules=["q_proj", "v_proj"],  # which weight matrices to adapt
    lora_dropout=0.05,
    bias="none",
)

print("=== FinGPT LoRA Configuration ===")
print(f"Base model: {base_model_id}")
print(f"LoRA rank: {lora_config.r}")
print(f"Target modules: {lora_config.target_modules}")
print(f"Trainable params: ~4M (vs. ~7B base params = 0.06% of total)")
print()

# Step 3: Financial instruction format
def format_financial_instruction(instruction: str, input_text: str, output: str = "") -> str:
    """FinGPT instruction format for fine-tuning."""
    prompt = f"""Instruction: {instruction}
Input: {input_text}
Answer: """
    if output:
        prompt += output
    return prompt

# Example training instances for FinGPT
training_examples = [
    {
        "instruction": "What is the sentiment of this financial news? Answer: positive, negative, or neutral.",
        "input": "Goldman Sachs raises its S&P 500 year-end target to 5,100 citing strong earnings growth.",
        "output": "positive"
    },
    {
        "instruction": "Summarize the key financial metric from this text.",
        "input": "The company reported Q3 revenue of $4.2B, up 18% YoY, with EBITDA margin expanding 200bps to 24%.",
        "output": "Q3 revenue: $4.2B (+18% YoY), EBITDA margin: 24% (+200bps)"
    },
]

for i, example in enumerate(training_examples):
    print(f"Training Example {i+1}:")
    formatted = format_financial_instruction(
        example['instruction'], example['input'], example['output']
    )
    print(formatted)
    print()

ModuleNotFoundError: No module named 'peft'

---
## 7. Multilingual LLMs

Most internet content is in English. Multilingual models extend NLP capabilities to the ~7,000 languages spoken globally with a strong focus on the top 200+ languages that have any significant digital presence.

### Major Multilingual LLMs

| Model | Languages | Key Innovation |
|-------|-----------|---------------|
| **Aya 8B / 35B** (Cohere) | 101 | Community-driven multilingual instruction data; Aya Collection (513M samples) |
| **BLOOMZ / mT0** | 46 | BLOOM + multilingual instruction tuning (xP3 dataset); best instruction-following |
| **XLM-R** (Meta) | 100 | RoBERTa trained on CC-100 filtered corpus; strong cross-lingual transfer |
| **NLLB-200** (Meta) | 200 | "No Language Left Behind" dedicated translation model; Flores-200 eval |
| **M2M-100** (Meta) | 100 | Many-to-many translation; all 100×100 language pairs (no English pivot) |
| **SeamlessM4T** (Meta) | 100+ | Speech + text; ASR, MT, TTS in one model |
| **mBERT** (Google) | 104 | Original multilingual BERT; shared vocabulary, Wikipedia training |

### Cross-Lingual Transfer Learning

XLM-R demonstrates a key property: a model trained on multilingual data can **transfer** to a new language with zero labeled data in that language (zero-shot cross-lingual transfer).

The key insight is that languages share structural patterns if we train a model on English NLI, it partially learns universal features that generalize to French, German, or Swahili.

$$\text{Zero-shot XLT: } \underbrace{\text{train}(\mathcal{D}_{en})}_{\text{labeled English}} \rightarrow \underbrace{\text{evaluate}(\mathcal{D}_{\text{target}})}_{\text{no labeled data}}$$

### NLLB-200 Translation

NLLB (No Language Left Behind) uses language tags in the format `{lang}_{script}` (e.g., `fra_Latn` for French, `hin_Deva` for Hindi). It was trained on FLORES-200, covering low-resource languages like Luganda, Wolof, and Fula.

### The Curse of Multilinguality

As more languages are added with fixed model capacity, performance on each individual language decreases there is a **capacity-performance trade-off**. Solutions include:
- Larger models (mDeBERTa, XLM-R XL/XXL)
- Language-specific adapter layers
- Mixture-of-Experts (MoE) routing by language

In [11]:
# Skip heavy model loading in automated execution
import os
if not os.environ.get("RUN_HEAVY_MODELS"):
    print("[Skipped: requires model download. Set RUN_HEAVY_MODELS=1 to run]")
else:
    # Multilingual NLP: NLLB-200 for translation + XLM-R for cross-lingual classification

    from transformers import pipeline, AutoTokenizer, AutoModel
    import torch

    # ── NLLB-200: Neural Machine Translation ─────────────────────────────────────
    # facebook/nllb-200-distilled-600M is the smallest variant (~2.4GB)
    # facebook/nllb-200-3.3B is the largest open model

    translator = pipeline(
        "translation",
        model="facebook/nllb-200-distilled-600M",
        device=0 if torch.cuda.is_available() else -1,
    )

    source_text = (
        "Artificial intelligence is transforming how we interact with information "
        "and make decisions in everyday life."
    )

    # NLLB language codes follow: {language}_{script}
    # Full list: https://github.com/facebookresearch/flores/blob/main/flores200/README.md
    target_languages = [
        ("French",    "fra_Latn"),
        ("Spanish",   "spa_Latn"),
        ("Hindi",     "hin_Deva"),
        ("Arabic",    "arb_Arab"),
        ("Swahili",   "swh_Latn"),
        ("Japanese",  "jpn_Jpan"),
    ]

    print("=== NLLB-200 Translations ===")
    print(f"Source (English): {source_text}\n")

    for lang_name, lang_code in target_languages:
        translation = translator(
            source_text,
            src_lang="eng_Latn",
            tgt_lang=lang_code,
            max_length=256
        )
        print(f"{lang_name} ({lang_code}):")
        print(f"  {translation[0]['translation_text']}")
        print()

[Skipped: requires model download. Set RUN_HEAVY_MODELS=1 to run]


In [12]:
# Skip heavy model loading in automated execution
import os
if not os.environ.get("RUN_HEAVY_MODELS"):
    print("[Skipped: requires model download. Set RUN_HEAVY_MODELS=1 to run]")
else:
    # XLM-R: Zero-shot cross-lingual text classification
    # We demonstrate how XLM-R embeddings enable cross-lingual similarity

    from transformers import AutoTokenizer, AutoModel
    import torch
    import torch.nn.functional as F

    xlmr_tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")
    xlmr_model = AutoModel.from_pretrained("xlm-roberta-base")
    xlmr_model.eval()

    def mean_pooling(model_output, attention_mask):
        """Average token embeddings (ignoring padding) for sentence representation."""
        token_embeddings = model_output.last_hidden_state
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
        return torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(
            input_mask_expanded.sum(1), min=1e-9
        )

    def get_xlmr_embedding(text: str) -> torch.Tensor:
        """Get normalized sentence embedding from XLM-R."""
        inputs = xlmr_tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=512)
        with torch.no_grad():
            outputs = xlmr_model(**inputs)
        embedding = mean_pooling(outputs, inputs['attention_mask'])
        return F.normalize(embedding, p=2, dim=1)

    # Same sentiment expressed in different languages
    sentences = {
        "EN (positive)": "The product is absolutely fantastic and exceeded all my expectations.",
        "FR (positive)": "Le produit est absolument fantastique et a dépassé toutes mes attentes.",
        "ES (positive)": "El producto es absolutamente fantástico y superó todas mis expectativas.",
        "EN (negative)": "This product is terrible and a complete waste of money.",
    }

    embeddings = {k: get_xlmr_embedding(v) for k, v in sentences.items()}

    print("=== XLM-R Cross-lingual Cosine Similarities ===")
    print("(Higher = more similar meaning)\n")

    keys = list(embeddings.keys())
    for i in range(len(keys)):
        for j in range(i + 1, len(keys)):
            sim = torch.nn.functional.cosine_similarity(
                embeddings[keys[i]], embeddings[keys[j]]
            ).item()
            print(f"  {keys[i]:<20} <-> {keys[j]:<20} : {sim:.4f}")

    print("\n→ Positive sentences across EN/FR/ES should cluster together (high similarity)")
    print("→ Positive vs. Negative should have low similarity even within English")

[Skipped: requires model download. Set RUN_HEAVY_MODELS=1 to run]


---
## 8. Reasoning LLMs

The latest generation of LLMs goes beyond pattern matching to perform **extended deliberate reasoning** before answering. This is often described using the analogy of System 1 vs. System 2 thinking from cognitive science.

### System 1 vs. System 2 Thinking

| | System 1 (Fast) | System 2 (Deliberate) |
|-|-----------------|----------------------|
| **Speed** | Instantaneous | Slow, sequential |
| **Nature** | Intuitive, automatic | Analytical, effortful |
| **LLM analog** | Standard next-token prediction | Extended chain-of-thought, self-correction |
| **Examples** | "What is 2+2?" | Multi-step math proof, debugging complex code |
| **Error type** | Bias, pattern shortcuts | More robust but can still err |

Reasoning LLMs allocate **"thinking tokens"** or internal scratchpad space before giving a final answer effectively performing System 2 reasoning.

### Major Reasoning LLMs

| Model | Organization | Approach |
|-------|-------------|----------|
| **o1 / o3 / o4-mini** | OpenAI | Internal chain-of-thought (hidden reasoning tokens); RLHF on thinking traces |
| **Gemini 2.5 Pro** | Google DeepMind | "Thinking" mode with visible reasoning; strongest on AIME/Olympiad |
| **Claude 3.7 Sonnet** | Anthropic | Extended thinking mode; explicit `<thinking>` blocks |
| **QwQ-32B** | Alibaba (Qwen) | Open-weight reasoning model; self-reflection and error correction |
| **DeepSeek-R1** | DeepSeek AI | Pure RL training (GRPO) on math/code; process reward model; open-weights |
| **Sky-T1** | NovaSky-Berkeley | Open reproduction of o1-like reasoning |

### DeepSeek-R1: Training with GRPO

DeepSeek-R1 uses **Group Relative Policy Optimization (GRPO)**, a reinforcement learning approach:

1. Sample $G$ outputs $\{o_1, ..., o_G\}$ from current policy $\pi_\theta$ for each question
2. Score each output with a reward function $r_i$ (correctness + format)
3. Compute group-relative advantage: $\hat{A}_i = \frac{r_i - \text{mean}(r)}{\text{std}(r)}$
4. Update policy to increase probability of high-advantage outputs

$$\mathcal{L}_{\text{GRPO}}(\theta) = -\mathbb{E}_{q, \{o_i\}} \left[ \frac{1}{G} \sum_{i=1}^{G} \min\left( \frac{\pi_\theta(o_i|q)}{\pi_{\text{ref}}(o_i|q)} \hat{A}_i, \; \text{clip}\left(\cdot, 1 \pm \epsilon\right) \hat{A}_i \right) \right]$$

**Key finding**: DeepSeek-R1 achieves reasoning ability comparable to OpenAI o1 on math/code benchmarks despite being fully open-source and trained with a simpler pipeline.

In [13]:
# Skip heavy model loading in automated execution
import os
if not os.environ.get("RUN_HEAVY_MODELS"):
    print("[Skipped: requires model download. Set RUN_HEAVY_MODELS=1 to run]")
else:
    # Reasoning LLMs: DeepSeek-R1 and QwQ usage patterns
    # These models use explicit <think>...</think> reasoning blocks

    # ── DeepSeek-R1 via transformers ──────────────────────────────────────────────
    # deepseek-ai/DeepSeek-R1-Distill-Qwen-7B is the smallest distilled variant
    # deepseek-ai/DeepSeek-R1 is the full 671B MoE model

    from transformers import AutoTokenizer, AutoModelForCausalLM
    import torch

    model_id = "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B"  # 7B distilled version

    tokenizer = AutoTokenizer.from_pretrained(model_id)
    # model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.bfloat16, device_map="auto")

    # DeepSeek-R1 uses a specific chat template
    # The model generates <think> blocks before answering

    hard_math_problem = (
        "In a right triangle, the two legs have lengths 5 and 12. "
        "A circle is inscribed in the triangle (touches all three sides). "
        "What is the radius of the inscribed circle?"
    )

    messages = [
        {"role": "user", "content": hard_math_problem}
    ]

    # Apply chat template
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    print("=== DeepSeek-R1 Prompt ===")
    print(prompt)

    # ── Simulated R1 reasoning output ─────────────────────────────────────────────
    simulated_output = """<think>
    Let me denote the legs as a=5, b=12. The hypotenuse c = sqrt(25 + 144) = sqrt(169) = 13.

    For an inscribed circle in a right triangle, the radius formula is:
    r = (a + b - c) / 2

    Let me verify this formula. The area of the triangle = (1/2) * a * b = (1/2)(5)(12) = 30.
    Also, Area = r * s where s is the semi-perimeter.
    s = (a + b + c) / 2 = (5 + 12 + 13) / 2 = 15

    So r = Area / s = 30 / 15 = 2.

    Cross-checking with the formula: r = (5 + 12 - 13) / 2 = 4 / 2 = 2. ✓
    </think>

    The radius of the inscribed circle is **r = 2**.

    **Solution:**
    - Legs: $a = 5$, $b = 12$  
    - Hypotenuse: $c = \\sqrt{5^2 + 12^2} = 13$
    - Area: $A = \\frac{1}{2}(5)(12) = 30$
    - Semi-perimeter: $s = \\frac{5+12+13}{2} = 15$
    - Inradius: $r = \\frac{A}{s} = \\frac{30}{15} = \\boxed{2}$
    """

    print("\n=== DeepSeek-R1 Output (with thinking) ===")
    print(simulated_output)

[Skipped: requires model download. Set RUN_HEAVY_MODELS=1 to run]


In [14]:
# ── Parsing reasoning traces from R1-style models ────────────────────────────
import re

def parse_reasoning_output(raw_output: str) -> dict:
    """
    Parse DeepSeek-R1 / QwQ style output into reasoning trace and final answer.
    
    Returns:
        dict with 'thinking' (str) and 'answer' (str) keys.
    """
    thinking_pattern = re.compile(r'<think>(.*?)</think>', re.DOTALL)
    thinking_match = thinking_pattern.search(raw_output)
    
    thinking = thinking_match.group(1).strip() if thinking_match else ""
    
    # Everything after the closing </think> tag is the answer
    if thinking_match:
        answer = raw_output[thinking_match.end():].strip()
    else:
        answer = raw_output.strip()
    
    return {"thinking": thinking, "answer": answer}

# Test with our simulated output
simulated_r1_output = """<think>
For a right triangle with legs a, b and hypotenuse c:
c = sqrt(a^2 + b^2) = sqrt(25+144) = 13
inradius r = (a + b - c) / 2 = (5 + 12 - 13) / 2 = 4/2 = 2
</think>

The radius of the inscribed circle is **r = 2**."""

parsed = parse_reasoning_output(simulated_r1_output)
print("=== Parsed Reasoning Output ===")
print(f"THINKING ({len(parsed['thinking'])} chars):")
print(parsed['thinking'])
print(f"\nFINAL ANSWER:")
print(parsed['answer'])

# ── vLLM for efficient inference of large reasoning models ────────────────────
print("\n=== vLLM: High-throughput Reasoning Model Inference ===")
vllm_example = """
# For DeepSeek-R1 (671B) or QwQ-32B, use vLLM for efficient serving:
# pip install vllm

from vllm import LLM, SamplingParams

llm = LLM(
    model="deepseek-ai/DeepSeek-R1-Distill-Qwen-32B",
    tensor_parallel_size=4,      # split across 4 GPUs
    max_model_len=32768,         # support long reasoning traces
    gpu_memory_utilization=0.90,
)

sampling_params = SamplingParams(
    temperature=0.6,
    top_p=0.95,
    max_tokens=8192,             # reasoning models need longer outputs
)

outputs = llm.generate([prompt], sampling_params)
print(outputs[0].outputs[0].text)
"""
print(vllm_example)

=== Parsed Reasoning Output ===
THINKING (152 chars):
For a right triangle with legs a, b and hypotenuse c:
c = sqrt(a^2 + b^2) = sqrt(25+144) = 13
inradius r = (a + b - c) / 2 = (5 + 12 - 13) / 2 = 4/2 = 2

FINAL ANSWER:
The radius of the inscribed circle is **r = 2**.

=== vLLM: High-throughput Reasoning Model Inference ===

# For DeepSeek-R1 (671B) or QwQ-32B, use vLLM for efficient serving:
# pip install vllm

from vllm import LLM, SamplingParams

llm = LLM(
    model="deepseek-ai/DeepSeek-R1-Distill-Qwen-32B",
    tensor_parallel_size=4,      # split across 4 GPUs
    max_model_len=32768,         # support long reasoning traces
    gpu_memory_utilization=0.90,
)

sampling_params = SamplingParams(
    temperature=0.6,
    top_p=0.95,
    max_tokens=8192,             # reasoning models need longer outputs
)

outputs = llm.generate([prompt], sampling_params)
print(outputs[0].outputs[0].text)



---
## 9. How to Fine-tune for a Domain

Building a domain-specific LLM does not require training from scratch (which costs millions of dollars). Three practical approaches exist, ordered by cost and impact:

### Approach A: Domain-Adaptive Pretraining (DAPT)

Continue pretraining a general model on domain-specific unlabeled text using the **standard language modeling objective**.

$$\mathcal{L}_{\text{LM}} = -\sum_{t} \log P(x_t \mid x_{<t}; \theta)$$

**When to use**: You have 1B+ tokens of domain text and want to fundamentally shift the model's vocabulary distribution and knowledge.

**Examples**: Meditron-70B (continued pretraining on medical literature), Llemma (math corpus), BloombergGPT (financial data).

**Cost**: ~$50K-$1M+ in GPU compute for a 7B-70B model.

### Approach B: Instruction Tuning (SFT)

Fine-tune on curated (instruction, response) pairs teaching the model **how to behave** in domain-specific interactions.

$$\mathcal{L}_{\text{SFT}} = -\sum_{t} \log P(y_t \mid x, y_{<t}; \theta) \quad \text{(only over response tokens)}$$

**When to use**: You have 1K-100K high-quality domain Q&A pairs, or can generate them with GPT-4 (synthetic data).

**Examples**: MetaMath (math instruction pairs), WizardCoder (code instruction pairs), FinGPT (financial instructions).

**Cost**: $50-$5,000 using LoRA/QLoRA on a single GPU.

### Approach C: RLHF / DPO

Use **preference optimization** to align model outputs with domain expert preferences.

**DPO (Direct Preference Optimization)** loss:

$$\mathcal{L}_{\text{DPO}}(\theta) = -\mathbb{E}_{(x, y_w, y_l)} \left[ \log \sigma \left( \beta \log \frac{\pi_\theta(y_w|x)}{\pi_{\text{ref}}(y_w|x)} - \beta \log \frac{\pi_\theta(y_l|x)}{\pi_{\text{ref}}(y_l|x)} \right) \right]$$

where $y_w$ is the preferred ("winner") response and $y_l$ is the rejected ("loser") response.

**When to use**: After SFT, to improve output quality with expert-labeled preferences.

### Data Quality Considerations

| Issue | Solution |
|-------|----------|
| **Duplicates** | MinHash LSH deduplication (document + paragraph level) |
| **Low quality** | Perplexity filtering (high perplexity = noisy text) |
| **Domain coverage** | Ensure representation across sub-domains (e.g., oncology + cardiology + neurology for medical) |
| **Temporal bias** | Balance time periods; avoid stale clinical guidelines |
| **PII/PHI** | De-identification pipeline before training |

### Evaluation Strategy

Always evaluate on **both** domain and general benchmarks to detect catastrophic forgetting:

```
Domain benchmarks:  MedQA, HumanEval, GSM8K, FinQA (domain-specific)
General benchmarks: MMLU, HellaSwag, TruthfulQA, BBH (general capability)
```

In [15]:
# Skip heavy model loading in automated execution
import os
if not os.environ.get("RUN_HEAVY_MODELS"):
    print("[Skipped: requires model download. Set RUN_HEAVY_MODELS=1 to run]")
else:
    # Domain Fine-tuning: Complete template using HuggingFace Trainer + LoRA/QLoRA
    # This is production-quality pseudo-code illustrating the full pipeline

    # ── Imports ───────────────────────────────────────────────────────────────────
    from dataclasses import dataclass
    from typing import Optional

    # Core training dependencies
    # pip install transformers datasets peft trl accelerate bitsandbytes
    from transformers import (
        AutoTokenizer,
        AutoModelForCausalLM,
        TrainingArguments,
        BitsAndBytesConfig,   # for QLoRA (4-bit quantization)
    )
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
    from trl import SFTTrainer, DataCollatorForCompletionOnlyLM
    from datasets import load_dataset
    import torch


    # ── Configuration ─────────────────────────────────────────────────────────────
    @dataclass
    class DomainFineTuneConfig:
        base_model: str = "meta-llama/Meta-Llama-3-8B"  # base model to adapt
        domain: str = "medical"                          # for naming and logging
        dataset_path: str = "./data/medical_instructions.jsonl"  # your domain data
        output_dir: str = "./models/llama3-medical"
    
        # LoRA hyperparameters
        lora_r: int = 16           # rank higher = more expressive, more VRAM
        lora_alpha: int = 32       # scaling = lora_alpha / lora_r
        lora_dropout: float = 0.05
    
        # QLoRA (4-bit quantization)
        use_qlora: bool = True     # set False to use regular LoRA (more VRAM)
    
        # Training
        num_train_epochs: int = 3
        per_device_train_batch_size: int = 2
        gradient_accumulation_steps: int = 8  # effective batch size = 16
        learning_rate: float = 2e-4
        max_seq_length: int = 2048
        warmup_ratio: float = 0.03


    def build_quantization_config(cfg: DomainFineTuneConfig) -> Optional[BitsAndBytesConfig]:
        """4-bit NF4 quantization for QLoRA (fits 70B in ~48GB VRAM)."""
        if not cfg.use_qlora:
            return None
        return BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,   # nested quantization for extra savings
            bnb_4bit_quant_type="nf4",        # NormalFloat4 better than Int4 for LLMs
            bnb_4bit_compute_dtype=torch.bfloat16,
        )


    def build_lora_config(cfg: DomainFineTuneConfig) -> LoraConfig:
        """LoRA configuration targeting attention and MLP layers."""
        return LoraConfig(
            r=cfg.lora_r,
            lora_alpha=cfg.lora_alpha,
            lora_dropout=cfg.lora_dropout,
            bias="none",
            task_type="CAUSAL_LM",
            # For Llama-3: target all projection matrices
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                             "gate_proj", "up_proj", "down_proj"],
        )


    def format_medical_instruction(example: dict) -> str:
        """
        Format a single medical instruction example into the model's chat template.
        Adjust this for your specific domain (legal, financial, scientific, etc.).
        """
        system_prompt = (
            "You are a knowledgeable medical assistant. "
            "Provide accurate, evidence-based medical information. "
            "Always recommend consulting a qualified healthcare provider."
        )
        return (
            f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n"
            f"{system_prompt}<|eot_id|>"
            f"<|start_header_id|>user<|end_header_id|>\n"
            f"{example['instruction']}<|eot_id|>"
            f"<|start_header_id|>assistant<|end_header_id|>\n"
            f"{example['output']}<|eot_id|>"
        )


    def run_domain_finetuning(cfg: DomainFineTuneConfig):
        """Main fine-tuning function DAPT step B (instruction tuning)."""

        # 1. Load tokenizer
        tokenizer = AutoTokenizer.from_pretrained(cfg.base_model, use_fast=True)
        tokenizer.pad_token = tokenizer.eos_token
        tokenizer.padding_side = "right"  # required for SFT with causal LM

        # 2. Load model (optionally quantized)
        bnb_config = build_quantization_config(cfg)
        model = AutoModelForCausalLM.from_pretrained(
            cfg.base_model,
            quantization_config=bnb_config,
            device_map="auto",
            torch_dtype=torch.bfloat16 if not cfg.use_qlora else None,
        )

        # 3. Prepare for k-bit training (QLoRA specific)
        if cfg.use_qlora:
            model = prepare_model_for_kbit_training(model)

        # 4. Apply LoRA adapters
        lora_config = build_lora_config(cfg)
        model = get_peft_model(model, lora_config)
        model.print_trainable_parameters()  # shows % of params being trained

        # 5. Load domain dataset
        dataset = load_dataset("json", data_files=cfg.dataset_path, split="train")
        # Format each example
        dataset = dataset.map(
            lambda x: {"text": format_medical_instruction(x)},
            remove_columns=dataset.column_names
        )

        # 6. Training arguments
        training_args = TrainingArguments(
            output_dir=cfg.output_dir,
            num_train_epochs=cfg.num_train_epochs,
            per_device_train_batch_size=cfg.per_device_train_batch_size,
            gradient_accumulation_steps=cfg.gradient_accumulation_steps,
            learning_rate=cfg.learning_rate,
            warmup_ratio=cfg.warmup_ratio,
            lr_scheduler_type="cosine",
            fp16=False,
            bf16=True,
            logging_steps=10,
            save_strategy="epoch",
            optim="paged_adamw_32bit",   # memory-efficient optimizer
            gradient_checkpointing=True,  # trade compute for memory
            report_to="wandb",           # track experiment with W&B
            run_name=f"{cfg.domain}-finetune",
        )

        # 7. SFT Trainer (only computes loss on completion tokens, not prompt)
        trainer = SFTTrainer(
            model=model,
            tokenizer=tokenizer,
            train_dataset=dataset,
            dataset_text_field="text",
            max_seq_length=cfg.max_seq_length,
            args=training_args,
        )

        # 8. Train
        trainer.train()

        # 9. Save LoRA adapters (small ~50MB for a 7B model)
        trainer.save_model(cfg.output_dir)
        print(f"\nLoRA adapters saved to: {cfg.output_dir}")
        print("To merge adapters into the base model for deployment:")
        print("  merged = model.merge_and_unload()")
        print("  merged.save_pretrained('./models/llama3-medical-merged')")


    # ── Demo: Print the config (actual training not executed here) ─────────────────
    cfg = DomainFineTuneConfig()
    print("=== Domain Fine-tuning Configuration ===")
    for field, value in cfg.__dict__.items():
        print(f"  {field:<40} = {value}")

[Skipped: requires model download. Set RUN_HEAVY_MODELS=1 to run]


In [16]:
# ── DPO Fine-tuning: Preference Optimization ──────────────────────────────────
# After SFT, use DPO to improve output quality with preference data

from trl import DPOTrainer, DPOConfig
from datasets import Dataset

# Example preference dataset structure for medical domain
# Each example has: prompt, chosen response (expert-preferred), rejected response
preference_examples = [
    {
        "prompt": "What are the first-line treatments for Type 2 diabetes?",
        "chosen": (
            "First-line treatment for Type 2 diabetes typically includes lifestyle "
            "modifications (diet, exercise, weight loss) combined with metformin "
            "pharmacotherapy, per ADA 2024 guidelines. Individualize based on "
            "kidney function, cardiovascular risk, and patient preference. "
            "Consult your endocrinologist for personalized management."
        ),
        "rejected": (
            "Just take metformin and watch what you eat. "
            "Diabetes is easy to manage with medication."
        )
    },
    {
        "prompt": "What does an elevated troponin level indicate?",
        "chosen": (
            "Elevated cardiac troponin (cTnI or cTnT) indicates myocardial injury. "
            "The differential includes acute MI (STEMI/NSTEMI), myocarditis, "
            "pulmonary embolism, stress cardiomyopathy, or demand ischemia. "
            "Serial troponins (0h/1h/3h) with clinical context are needed for diagnosis."
        ),
        "rejected": (
            "Elevated troponin means you are having a heart attack. "
            "Go to the emergency room immediately."
        )
    }
]

# Create dataset in TRL DPO format
dpo_dataset = Dataset.from_list(preference_examples)
print("=== DPO Preference Dataset ===")
print(f"Number of preference pairs: {len(dpo_dataset)}")
print(f"Columns: {dpo_dataset.column_names}")

# DPO training setup (pseudo-code model loaded separately)
print("\n=== DPO Training Configuration ===")
dpo_config = DPOConfig(
    output_dir="./models/llama3-medical-dpo",
    beta=0.1,           # KL divergence strength; lower = more deviation from SFT
    num_train_epochs=1, # DPO converges faster than SFT
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=5e-7, # much lower LR than SFT
    bf16=True,
)

print(f"DPO beta (KL regularization):  {dpo_config.beta}")
print(f"Learning rate:                 {dpo_config.learning_rate}")
print(f"Note: DPO requires both the trained model AND the reference (SFT) model")
print(f"The reference model is frozen; KL divergence prevents over-optimization.")

ModuleNotFoundError: No module named 'trl'

In [17]:
# ── Evaluation: Domain-specific + General benchmarks ─────────────────────────
# Use lm-evaluation-harness for standardized benchmark evaluation
# pip install lm-eval

# Example: Evaluating a medical LLM on MedQA-USMLE and general MMLU

import subprocess

def build_eval_command(
    model_path: str,
    tasks: list,
    num_fewshot: int = 5,
    batch_size: int = 16,
    output_path: str = "./eval_results"
) -> str:
    """Build lm-evaluation-harness command for benchmarking."""
    task_str = ",".join(tasks)
    return (
        f"lm_eval "
        f"--model hf "
        f"--model_args pretrained={model_path},dtype=bfloat16 "
        f"--tasks {task_str} "
        f"--num_fewshot {num_fewshot} "
        f"--batch_size {batch_size} "
        f"--output_path {output_path} "
        f"--log_samples"
    )

# Domain-specific evaluation tasks
domain_tasks = [
    "medqa_4options",      # USMLE-style medical QA
    "pubmedqa",            # biomedical research QA
    "mmlu_anatomy",        # MMLU anatomy subset
    "mmlu_clinical_knowledge",
    "mmlu_medical_genetics",
]

# General capability tasks (check for catastrophic forgetting)
general_tasks = [
    "mmlu",           # massive multitask language understanding
    "hellaswag",      # commonsense NLI
    "truthfulqa_mc1", # factual accuracy
    "gsm8k",          # arithmetic reasoning
]

model_path = "./models/llama3-medical-merged"

print("=== Evaluation Commands ===")
print("\n# Domain-specific benchmarks:")
print(build_eval_command(model_path, domain_tasks, num_fewshot=5))

print("\n# General capability benchmarks (catastrophic forgetting check):")
print(build_eval_command(model_path, general_tasks, num_fewshot=5))

# Expected results table
print("\n=== Example Results Comparison ===")
print(f"{'Benchmark':<30} {'Base LLaMA-3-8B':>18} {'Medical FT':>12} {'Delta':>8}")
print("-" * 70)
results = [
    ("MedQA (USMLE)",     "55.2",  "71.4",  "+16.2"),
    ("PubMedQA",          "61.0",  "74.8",  "+13.8"),
    ("MMLU Clinical",     "58.3",  "70.1",  "+11.8"),
    ("MMLU (all)",        "66.6",  "64.9",  "-1.7 "),  # slight forgetting
    ("HellaSwag",         "82.1",  "81.8",  "-0.3 "),
    ("TruthfulQA",        "43.2",  "44.1",  "+0.9 "),
    ("GSM8K",             "79.6",  "77.3",  "-2.3 "),  # some forgetting
]
for task, base, ft, delta in results:
    print(f"{task:<30} {base:>18} {ft:>12} {delta:>8}")

print("\n→ Domain FT improves medical tasks significantly")
print("→ Minor degradation on general tasks (acceptable trade-off)")
print("→ Use continual learning techniques to minimize forgetting")

=== Evaluation Commands ===

# Domain-specific benchmarks:
lm_eval --model hf --model_args pretrained=./models/llama3-medical-merged,dtype=bfloat16 --tasks medqa_4options,pubmedqa,mmlu_anatomy,mmlu_clinical_knowledge,mmlu_medical_genetics --num_fewshot 5 --batch_size 16 --output_path ./eval_results --log_samples

# General capability benchmarks (catastrophic forgetting check):
lm_eval --model hf --model_args pretrained=./models/llama3-medical-merged,dtype=bfloat16 --tasks mmlu,hellaswag,truthfulqa_mc1,gsm8k --num_fewshot 5 --batch_size 16 --output_path ./eval_results --log_samples

=== Example Results Comparison ===
Benchmark                         Base LLaMA-3-8B   Medical FT    Delta
----------------------------------------------------------------------
MedQA (USMLE)                                55.2         71.4    +16.2
PubMedQA                                     61.0         74.8    +13.8
MMLU Clinical                                58.3         70.1    +11.8
MMLU (all)       

---
## 10. Summary: Choosing the Right Specialized LLM

### Decision Framework

```
What is your domain?
├── Code generation/completion    → DeepSeek-Coder-V2, Code Llama, CodeGemma
├── Mathematical reasoning        → DeepSeek-Math, Qwen2.5-Math, Llemma
├── Medical/biomedical            → Meditron-70B, PubMedBERT, BioGPT
├── Legal documents               → SaulLM-54B, Legal-BERT
├── Scientific literature         → SciBERT, Galactica, OLMo
├── Financial analysis            → BloombergGPT (closed), FinGPT (open), FinBERT
├── Multilingual                  → Aya-35B, NLLB-200, XLM-R, SeamlessM4T
└── Complex multi-step reasoning  → DeepSeek-R1, o1/o3, QwQ-32B
```

### Open vs. Closed Models

| Consideration | Open Weights | Closed API |
|--------------|-------------|------------|
| Data privacy | Full control | Data sent to provider |
| Customization | Fine-tune freely | Limited (fine-tuning APIs) |
| Cost at scale | Hardware CAPEX | Per-token OPEX |
| Maintenance | Self-managed | Provider-managed |
| Cutting-edge capability | Slightly behind | Frontier |

### Key Takeaways

1. **Domain data is a competitive moat** a 7B model trained on curated domain data can outperform a 70B general model
2. **LoRA/QLoRA democratizes fine-tuning** a single A100 GPU can fine-tune a 70B model in a weekend
3. **Reasoning models represent a paradigm shift** thinking tokens enable qualitatively different problem-solving
4. **Evaluation requires both domain and general benchmarks** catch catastrophic forgetting early
5. **RAG + specialized model** is often better than either alone combine structured knowledge retrieval with domain-tuned generation

---
## 11. Additional Learning Resources

### Foundational Papers

#### Code LLMs
- **Code Llama**: Rozière et al. (2023). *Code Llama: Open Foundation Models for Code*. https://arxiv.org/abs/2308.12950
- **DeepSeek-Coder**: Guo et al. (2024). *DeepSeek-Coder: When the Large Language Model Meets Programming*. https://arxiv.org/abs/2401.14196
- **StarCoder**: Li et al. (2023). *StarCoder: May the Source Be with You!*. https://arxiv.org/abs/2305.06161
- **AlphaCode 2**: DeepMind (2023). *AlphaCode 2 Technical Report*. https://storage.googleapis.com/deepmind-media/AlphaCode2/AlphaCode2_Tech_Report.pdf

#### Math LLMs
- **Llemma**: Azerbayev et al. (2023). *Llemma: An Open Language Model for Mathematics*. https://arxiv.org/abs/2310.10631
- **MetaMath**: Yu et al. (2023). *MetaMath: Bootstrap Your Own Mathematical Questions for Large Language Models*. https://arxiv.org/abs/2309.12284
- **DeepSeek-Math**: Shao et al. (2024). *DeepSeekMath: Pushing the Limits of Mathematical Reasoning in Open Language Models*. https://arxiv.org/abs/2402.03300

#### Medical LLMs
- **Med-PaLM 2**: Singhal et al. (2023). *Towards Expert-Level Medical Question Answering with Large Language Models*. https://arxiv.org/abs/2305.09617
- **Meditron**: Chen et al. (2023). *MEDITRON-70B: Scaling Medical Pretraining for Large Language Models*. https://arxiv.org/abs/2311.16079

#### Financial LLMs
- **BloombergGPT**: Wu et al. (2023). *BloombergGPT: A Large Language Model for Finance*. https://arxiv.org/abs/2303.17564
- **FinGPT**: Yang et al. (2023). *FinGPT: Open-Source Financial Large Language Models*. https://arxiv.org/abs/2306.06031

#### Multilingual LLMs
- **Aya**: Üstün et al. (2024). *Aya Model: An Instruction Finetuned Open-Access Multilingual Language Model*. https://arxiv.org/abs/2402.07827
- **NLLB-200**: NLLB Team (2022). *No Language Left Behind: Scaling Human-Centered Machine Translation*. https://arxiv.org/abs/2207.04672
- **XLM-R**: Conneau et al. (2020). *Unsupervised Cross-lingual Representation Learning at Scale*. https://arxiv.org/abs/1911.02116

#### Reasoning LLMs
- **DeepSeek-R1**: DeepSeek-AI (2025). *DeepSeek-R1: Incentivizing Reasoning Capability in LLMs via Reinforcement Learning*. https://arxiv.org/abs/2501.12948
- **OpenAI o1**: OpenAI (2024). *Learning to Reason with LLMs*. https://openai.com/index/learning-to-reason-with-llms/

#### Fine-tuning
- **LoRA**: Hu et al. (2021). *LoRA: Low-Rank Adaptation of Large Language Models*. https://arxiv.org/abs/2106.09685
- **QLoRA**: Dettmers et al. (2023). *QLoRA: Efficient Finetuning of Quantized LLMs*. https://arxiv.org/abs/2305.14314
- **DPO**: Rafailov et al. (2023). *Direct Preference Optimization: Your Language Model is Secretly a Reward Model*. https://arxiv.org/abs/2305.18290

### HuggingFace Model Collections

- **Code**: https://huggingface.co/collections/bigcode (StarCoder, SantaCoder)
- **Medical**: https://huggingface.co/microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract
- **Legal**: https://huggingface.co/nlpaueb/legal-bert-base-uncased
- **Financial**: https://huggingface.co/ProsusAI/finbert
- **Chemistry**: https://huggingface.co/seyonec/ChemBERTa-zinc-base-v1
- **Multilingual**: https://huggingface.co/facebook/nllb-200-distilled-600M
- **Reasoning**: https://huggingface.co/deepseek-ai/DeepSeek-R1

### Benchmark Leaderboards

| Benchmark | URL |
|-----------|-----|
| HumanEval / Code | https://evalplus.github.io/leaderboard.html |
| SWE-bench | https://www.swebench.com |
| MATH / AIME | https://huggingface.co/spaces/HuggingFaceH4/open-llm-leaderboard |
| Medical USMLE | https://huggingface.co/spaces/openlifescienceai/open_medical_llm_leaderboard |
| LexGLUE (Legal) | https://github.com/coastalcph/lex-glue |
| FinQA | https://github.com/czyssrs/FinQA |
| General (MMLU, etc.) | https://huggingface.co/spaces/HuggingFaceH4/open-llm-leaderboard |

### Key Tools

- **lm-evaluation-harness** (EleutherAI): Standardized LLM benchmarking https://github.com/EleutherAI/lm-evaluation-harness
- **TRL** (HuggingFace): SFT + DPO + PPO training library https://github.com/huggingface/trl
- **PEFT** (HuggingFace): LoRA, prefix tuning, adapter methods https://github.com/huggingface/peft
- **vLLM**: High-throughput LLM inference https://github.com/vllm-project/vllm
- **Axolotl**: All-in-one fine-tuning framework https://github.com/axolotl-ai-cloud/axolotl